# 역진자 행동 복제 — 학습 데이터 수에 따른 성능 비교

`SwingUp` 제어기를 교사로 두고 학습한 신경망이, **학습 데이터를 얼마나 쓰느냐**에 따라
얼마나 달라지는지 비교한다.

| 순서 | 하는 일 | 걸리는 시간 |
|---|---|---|
| 1 | 저장소 받기 | 10초 |
| 2 | 원본(206판) 모델 보관 | 즉시 |
| 3 | 40·60·80판으로 각각 수집 + 학습 | 약 8분 |
| 4 | 네 모델을 같은 조건에서 시뮬레이션 | 약 1분 |
| 5 | 120개 조건에서 성공률 측정 | 약 25분 |

**위에서부터 순서대로** 실행한다. 중간에 런타임이 끊기면 1번부터 다시 해야 한다.

> GPU는 필요 없다. 신경망이 작고 시간의 대부분이 물리 시뮬레이션이라 CPU로 돈다.

---
## 1. 저장소 받기

코드와 학습된 체크포인트가 모두 GitHub에 있어서 받아오기만 하면 된다.

- 시뮬레이션·평가는 **numpy** 만 있으면 돌고, 학습에는 **torch** 가 필요하다
- Colab에는 둘 다 기본으로 깔려 있다

In [1]:
import os
if not os.path.exists("/content/Pendulum_control"):
    !git clone -q https://github.com/jooyongsim/Pendulum_control.git
os.chdir("/content/Pendulum_control")       # 몇 번을 눌러도 안전하다
print("작업 폴더:", os.getcwd())
!ls behavior_cloning_yeen

작업 폴더: /content/Pendulum_control
'behavior_cloning_yeen — 역진자 swing-up 행동.pdf'   data	    __pycache__
 checkpoints					     eval_bc.py     README.md
 collect_expert.py				     figs	    train_bc.py
 dagger.py					     il_common.py


---
## 2. 원본 모델 보관

`checkpoints/bc_h5.npz` 는 **206판으로 학습된 원본**이다.
아래 학습 셀이 **같은 이름으로 덮어쓰므로**, 먼저 따로 빼둬야 나중에 비교할 수 있다.

`rename()` 은 체크포인트 안의 표시 이름을 바꾼다. 안 바꾸면 평가 출력에서
모든 모델이 `bc_h5` 로 똑같이 나와 구분이 안 된다.

In [2]:
import os, json, pathlib
import numpy as np
os.chdir("/content/Pendulum_control")

SWEEP = pathlib.Path("/content/sweep"); SWEEP.mkdir(exist_ok=True)

def rename(path, newname):
    """체크포인트 안의 표시 이름을 바꾼다 (평가 출력에서 구분하려고)."""
    d = dict(np.load(path, allow_pickle=False))
    meta = json.loads(str(d["meta_json"])); meta["name"] = newname
    d["meta_json"] = json.dumps(meta)
    np.savez_compressed(path, **d)

def info(path):
    """(학습 판수, 전체 RMSE, swing RMSE) 를 꺼낸다."""
    r = json.loads(str(np.load(path, allow_pickle=False)["meta_json"]))["report"]
    return r["episodes"], r["rmse"], r["rmse_swing"]

!cp behavior_cloning_yeen/checkpoints/bc_h5.npz {SWEEP}/bc_206.npz
rename(f"{SWEEP}/bc_206.npz", "ep206")
ep, rmse, sw = info(f"{SWEEP}/bc_206.npz")
print(f"원본 보관 완료 — {ep}판 학습, RMSE {rmse:.3f}, swing {sw:.3f}")

원본 보관 완료 — 206판 학습, RMSE 2.070, swing 3.451


---
## 3. 데이터 수를 바꿔가며 학습

`SIZES` 의 각 값마다 세 가지를 한다.

1. **수집** — 교사를 그만큼 돌려 매 0.01초의 `(상태, 행동)` 을 기록한다.
   판마다 마찰·팔 힘·지연을 무작위로 바꾸고, 70% 확률로 외란을 넣어
   **넘어졌다 다시 올리는 장면**도 데이터에 담는다
2. **학습** — 그 데이터로 신경망을 맞춘다 (80 epoch)
3. **보관** — 다음 반복이 덮어쓰기 전에 따로 저장한다

> ⚠️ **넣은 값보다 실제 채택 판 수가 적다.** 교사조차 한 번도 올리지 못한 판은
> 가르칠 정답이 없어 버리기 때문이다. 대략 80% 정도가 남는다.
> (예: `--episodes 40` → 31판)

In [3]:
import os; os.chdir("/content/Pendulum_control")

SIZES = [40, 60, 80]          # 원하는 값으로 바꿔도 된다

for n in SIZES:
    print(f"\n{'='*62}\n  {n}판 수집 → 학습\n{'='*62}")
    data = f"behavior_cloning_yeen/data/expert_{n}.npz"
    !PYTHONPATH=behavior_cloning_yeen python behavior_cloning_yeen/collect_expert.py --episodes {n} --out {data}
    !PYTHONPATH=behavior_cloning_yeen python behavior_cloning_yeen/train_bc.py --data {data} --history 5
    !cp behavior_cloning_yeen/checkpoints/bc_h5.npz {SWEEP}/bc_{n}.npz
    kept = info(f"{SWEEP}/bc_{n}.npz")[0]
    rename(f"{SWEEP}/bc_{n}.npz", f"ep{kept}")
    print(f"  보관 → {SWEEP}/bc_{n}.npz   (실제 {kept}판)")

print("\n보관된 모델")
for p in sorted(SWEEP.glob("*.npz"), key=lambda q: info(q)[0]):
    e, r, s = info(p)
    print(f"  {p.name:<12} {e:>4}판   RMSE {r:.3f}   swing {s:.3f}")


  40판 수집 → 학습
    25/40 episodes,   30020 samples, 5 dropped,  14.9s

31 episodes kept (9 expert failures dropped), 46531 samples -> behavior_cloning_yeen/data/expert_40.npz
  swing    11432 (24.6%)  u in [ -47.1,  +47.1]  std 24.00
  balance  35099 (75.4%)  u in [ -47.1,  +47.1]  std  6.06
  saturated |u| = u_max:  1.9%
  first catch  median 2.39 s
  episodes ending upright: 22/31
  collection took 24.1 s

--- history 5 --------------------------------------------------
    epoch    8/80  train 0.24757  val 0.16439
    epoch   16/80  train 0.20384  val 0.14002
    epoch   24/80  train 0.16476  val 0.10566
    epoch   32/80  train 0.14972  val 0.09083
    epoch   40/80  train 0.13377  val 0.09309
    epoch   48/80  train 0.11212  val 0.07779
    epoch   56/80  train 0.10597  val 0.07555
    epoch   64/80  train 0.09963  val 0.07174
    epoch   72/80  train 0.09727  val 0.07107
    epoch   80/80  train 0.09619  val 0.07066
  h=5  in=25  params=72705   RMSE  3.484  swing  6.444  balance

---
## 4. 시뮬레이션 — 네 모델을 같은 조건에서

모든 모델을 **똑같은 초기조건**에서 8초씩 돌린다.
마찰·팔 힘·시작 기울기가 전부 같으므로, 차이가 나면 그건 **오직 모델 때문**이다.

화면 구성:

- **위** — 진자 네 개가 동시에 움직인다. 왼쪽부터 데이터가 적은 순서.
  초록 부채꼴이 **직립 판정 구간(±10°)** 이고, 그 안에 들어오면 막대 끝이 초록으로 바뀐다
- **가운데** — 네 궤적을 겹친 그래프. 누가 먼저 초록 띠에 들어가 머무는지 보인다
- **아래** — 결과 표

> Colab에 한글 글꼴이 없어서 그래프 라벨만 영문으로 쓴다. 표는 HTML이라 한글이 정상이다.

In [4]:
import os, math, sys
if os.path.basename(os.getcwd()) != "Pendulum_control":
    os.chdir("/content/Pendulum_control")

import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display

sys.path.insert(0, "behavior_cloning_yeen")
import il_common as il
rm = il.rm

DURATION, EVERY, BAND = 8.0, 4, 10.0          # 8초, 4프레임마다, 직립 판정 ±10도
COLORS = ["#C9D9E6", "#8FB3CC", "#3E7CB1", "#E07A3F", "#8E5A9E", "#6B7A85"]
paths = sorted(SWEEP.glob("*.npz"), key=lambda p: info(p)[0])

runs, rows = [], []
for i, p in enumerate(paths):
    pol = il.BCPolicy(str(p)); rep = pol.meta["report"]
    T, TH, PH = [], [], []
    with il.model_errors(1.0, None):
        lp = il.make_loop(pol)                # 인자 없음 = 모든 모델 동일 초기조건
        while lp.plant.t < DURATION and not lp.stop:
            lp.advance(10)                    # 10 ms = 제어 1스텝
            T.append(lp.plant.t)
            TH.append(math.degrees(rm.wrap(lp.plant.x[0])))
            PH.append(math.degrees(lp.plant.x[2]))
    T, TH, PH = np.array(T), np.array(TH), np.array(PH)
    up = np.abs(TH) < BAND
    hold = float(np.mean(np.abs(TH[T >= T[-1] - 5.0]) < BAND) * 100)
    runs.append(dict(label=f"{rep['episodes']} ep", color=COLORS[i % len(COLORS)],
                     T=T, TH=TH, PH=PH))
    rows.append({"모델": f"{rep['episodes']}판", "학습 판수": rep["episodes"],
                 "swing RMSE": rep["rmse_swing"],
                 "처음 직립 [s]": float(T[np.argmax(up)]) if up.any() else np.nan,
                 "마지막 5초 직립 [%]": hold,
                 "마지막 각도 [deg]": float(TH[-1]),
                 "팔 최대 [deg]": float(np.abs(PH).max()),
                 "중단": lp.stop or "-"})

idx = np.arange(0, max(len(r["T"]) for r in runs), EVERY)
Tref = max(runs, key=lambda r: len(r["T"]))["T"]
n = len(runs)

fig = plt.figure(figsize=(11.5, 5.8), dpi=70)
L, lim = rm.L, 1.34 * rm.L
w = 0.96 / n
panels = []
for i, r in enumerate(runs):
    ax = fig.add_axes([0.02 + i * w, 0.50, w * 0.92, 0.46])
    ax.set_aspect("equal"); ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
    ax.set_xticks([]); ax.set_yticks([])
    ax.add_patch(Wedge((0, 0), L * 1.06, 90 - BAND, 90 + BAND,
                       facecolor="#2E8B57", alpha=0.16, edgecolor="none"))
    for s in (1, -1):
        a = math.radians(BAND)
        ax.plot([0, s * L * 1.06 * math.sin(a)], [0, L * 1.06 * math.cos(a)],
                color="#2E8B57", lw=0.9, ls=":")
    ax.add_patch(plt.Circle((0, 0), L, fill=False, color="0.9", ls="--", lw=.7))
    rod, = ax.plot([], [], lw=6, color=r["color"], solid_capstyle="round")
    bob, = ax.plot([], [], "o", ms=8, color=r["color"])
    ax.plot([0], [0], "ko", ms=4)
    ax.set_title(r["label"], fontsize=12, weight="bold", color=r["color"])
    txt = ax.text(0, -1.24 * L, "", ha="center", fontsize=9, family="monospace")
    panels.append((rod, bob, txt))

ax_th = fig.add_axes([0.06, 0.09, 0.90, 0.33])
lines, heads = [], []
for r in runs:
    ln, = ax_th.plot([], [], lw=1.6, color=r["color"], label=r["label"])
    hd, = ax_th.plot([], [], "o", ms=4, color=r["color"])
    lines.append(ln); heads.append(hd)
ax_th.axhspan(-BAND, BAND, color="#2E8B57", alpha=.16)
ax_th.text(0.15, 0, "upright band", fontsize=8, color="#2E8B57", va="center")
ax_th.set_xlim(0, Tref[-1]); ax_th.set_ylim(-190, 190)
ax_th.set_yticks([-180, -90, 0, 90, 180])
ax_th.set_xlabel("t [s]", fontsize=9); ax_th.set_ylabel("theta from up [deg]", fontsize=9)
ax_th.tick_params(labelsize=8); ax_th.grid(alpha=.3)
ax_th.legend(ncol=n, fontsize=9, loc="lower right")

def frame(k):
    out = []
    for (rod, bob, txt), ln, hd, r in zip(panels, lines, heads, runs):
        i = min(idx[k], len(r["T"]) - 1)
        th = math.radians(r["TH"][i]); x, y = L * math.sin(th), L * math.cos(th)
        inside = abs(r["TH"][i]) < BAND
        rod.set_data([0, x], [0, y]); bob.set_data([x], [y])
        bob.set_color("#2E8B57" if inside else r["color"])
        txt.set_text(f"{r['TH'][i]:+6.1f} deg"); txt.set_color("#2E8B57" if inside else "0.45")
        ln.set_data(r["T"][:i+1], r["TH"][:i+1]); hd.set_data([r["T"][i]], [r["TH"][i]])
        out += [rod, bob, txt, ln, hd]
    return out

ani = FuncAnimation(fig, frame, frames=len(idx), interval=40, blit=True)
plt.close(fig)
display(HTML(ani.to_jshtml()))

def hold_css(s):
    return ["color:#1C6B45; font-weight:700" if v >= 95
            else "color:#C0392B; font-weight:700" for v in s]

df = pd.DataFrame(rows).set_index("모델")
display(df.style
    .format({"swing RMSE": "{:.3f}", "처음 직립 [s]": "{:.2f}",
             "마지막 5초 직립 [%]": "{:.0f}%", "마지막 각도 [deg]": "{:+.1f}",
             "팔 최대 [deg]": "{:.1f}"})
    .background_gradient(cmap="Oranges", subset=["swing RMSE"], vmin=0, vmax=11)
    .background_gradient(cmap="Oranges", subset=["팔 최대 [deg]"], vmin=0, vmax=230)
    .apply(hold_css, subset=["마지막 5초 직립 [%]"])
    .highlight_min(subset=["swing RMSE", "처음 직립 [s]", "팔 최대 [deg]"],
                   props="border:2px solid #1C6B45;")
    .set_caption("같은 조건에서 1판씩 · 주황이 진할수록 나쁨 · 초록 테두리 = 최솟값 · "
                 "마지막 5초 직립이 95% 이상이면 성공")
    .set_properties(**{"text-align": "right", "padding": "6px 14px"})
    .set_table_styles([
        {"selector": "caption", "props": "caption-side:bottom; font-size:0.82em; "
                                         "color:#667; padding-top:10px; text-align:left;"},
        {"selector": "th", "props": "text-align:right; padding:6px 14px;"},
        {"selector": "tbody tr:nth-child(odd)", "props": "background-color:#ffffff;"}]))

Output hidden; open in https://colab.research.google.com to view.

---
## 5. 120개 조건에서 성공률 측정

위 시뮬레이션은 **한 판**이라 운이 섞인다. 조건을 120가지로 바꿔 각 모델을 돌리고
성공률을 센다. 이게 믿을 수 있는 지표다.

**성공** = 중단되지 않았고 + 마지막 5초의 95% 이상을 직립 ±10° 안에서 유지

출력 읽는 법:

```
ep65   up 110/120 (91.7%)  caught 100.0%  catch 2.44s  theta rms 0.99d  arm 61.4d  sat 0.6%  ARM LIMIT x8
        └성공률            └올린 비율      └세운 시각   └흔들림        └팔 최대          └중단 사유
```

> ⏱ 약 25분. 탭을 닫으면 중간에 끊긴다.

In [5]:
import os; os.chdir("/content/Pendulum_control")

CKPTS = " ".join(str(p) for p in sorted(SWEEP.glob("*.npz"), key=lambda p: info(p)[0]))
print("평가 대상:", CKPTS, "\n")

!PYTHONPATH=behavior_cloning_yeen python behavior_cloning_yeen/eval_bc.py \
    --episodes 120 --suite clean --ckpt {CKPTS}

평가 대상: /content/sweep/bc_40.npz /content/sweep/bc_60.npz /content/sweep/bc_80.npz /content/sweep/bc_206.npz 

=== swing-up only (no disturbance): 120 randomised plants (seed 1234), 15 s each
    success = within 10 deg of upright for >95% of the last 5 s

    expert       up 119/120 ( 99.2%)  caught 100.0%  catch  2.41s  theta rms 0.90d  arm  56.5d  sat  1.1%  ARM LIMIT (140 deg) x1
    ep31         up   0/120 (  0.0%)  caught 100.0%  catch  3.02s  theta rms   --   arm  76.3d  sat  0.7%  ARM LIMIT (140 deg) x4   | open-loop RMSE 3.484
    ep49         up 104/120 ( 86.7%)  caught 100.0%  catch  2.98s  theta rms 0.97d  arm  66.9d  sat  0.5%  -   | open-loop RMSE 3.318
    ep65         up  95/120 ( 79.2%)  caught 100.0%  catch  2.96s  theta rms 1.00d  arm  65.9d  sat  0.5%  ARM LIMIT (140 deg) x1   | open-loop RMSE 3.755
    ep206        up 111/120 ( 92.5%)  caught 100.0%  catch  2.41s  theta rms 1.03d  arm  62.5d  sat  0.6%  ARM LIMIT (140 deg) x8   | open-loop RMSE 2.070



---
## 결과를 읽을 때 주의할 것

**`swing RMSE` 로 모델을 고르면 안 된다.**
이 값은 *교사 궤적 위에서* 교사 명령과 얼마나 다른지만 잰다.
학생이 혼자 운전하면 교사가 안 가본 상태로 밀려나는데, 거기서의 실력은 이 점수에 안 잡힌다.

실제로 이 프로젝트의 DAgger 모델은 **RMSE가 더 나쁜데 성공률은 더 높다.**

| | RMSE | 120판 성공률 |
|---|---|---|
| `bc_h5` (206판) | 2.070 | 92.5% |
| `dagger_i3` | 3.360 | 95.8% |

믿을 지표는 **성공률** 과 **마지막 5초 직립 비율** 이다.

---

## 결과물 저장

Colab 런타임이 끊기면 전부 사라진다. 남길 것이 있으면 받아두자.

- 왼쪽 **📁 폴더 아이콘** → `sweep` 폴더 → 파일 우클릭 → 다운로드
- 출력된 표와 성적표는 **화면 캡처**
- 노트북 자체는 **파일 → Drive에 사본 저장**